# Day 2: Practical Python for JavaScript & TypeScript Developers

Welcome to Day 2 of **Learn Python in 5 Days**.

## What You Will Learn Today
- How to structure Python code with **Modules, Imports, and Packages**
- Object-Oriented fundamentals in Python: classes, `__init__`, and `self`
- Clean, boilerplate-free data modeling with **`@dataclass`**
- Modern object-oriented filesystem operations using **`pathlib`**
- Robust **JSON serialization & persistence** (`dump`, `load`, `dumps`, `loads`)
- Guaranteed resource safety with **Context Managers (`with`)**
- Granular **Exception Handling** and custom domain exceptions
- Variadic function arguments: **`*args` and `**kwargs`**
- Conceptual mechanics and practical uses of **Decorators**
- Practice Challenge: Refactoring the Task management system into a structured 3-layer architecture

---

## 1. Modules and Standard Library Imports
In Python, any `.py` file is a module. Python ships with a massive "batteries-included" standard library.

### JavaScript Comparison:
- **JS (ESM):** `import { readFileSync } from 'fs'; export const myVar = 10;`
- **Python:** `from pathlib import Path; my_var = 10` *(every top-level definition is automatically exportable)*

In [ ]:
# Standard library imports
import math
import json
from datetime import datetime, timezone
from pathlib import Path

current_time = datetime.now(timezone.utc)
print("Current UTC Time:", current_time.isoformat())
print("Pi rounded to 4 decimals:", f"{math.pi:.4f}")

## 2. Classes & Object-Oriented Fundamentals
Python classes use explicit **`self`** as the first parameter of methods. `__init__` is the constructor method.

> **Mental Model:** `self` is Python's explicit equivalent of JavaScript's `this`. Unlike JS, where `this` can detach when passing callbacks, Python bound methods permanently retain their instance reference.

In [ ]:
class LegacyTaskManager:
    def __init__(self, project_name: str):
        # Instance attributes
        self.project_name = project_name
        self.tasks = []

    def add_task(self, title: str) -> None:
        self.tasks.append(title)

    def get_count(self) -> int:
        return len(self.tasks)

# Instantiation (Note: No 'new' keyword in Python!)
manager = LegacyTaskManager("Alpha Project")
manager.add_task("Setup CI/CD pipeline")
manager.add_task("Configure CORS headers")

print(f"Project '{manager.project_name}' has {manager.get_count()} tasks:", manager.tasks)

## 3. Dataclasses (`@dataclass`)
Writing plain Python classes for simple data holding requires tedious `__init__`, `__repr__`, and `__eq__` boilerplate. 

**`@dataclass`** (introduced in Python 3.7) auto-generates all of these methods based on type annotations!

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Task:
    id: int
    title: str
    completed: bool = False
    priority: str = "Normal"
    tags: list[str] = field(default_factory=list)

# 1. Clean instantiation
t1 = Task(id=1, title="Implement OAuth login", priority="High", tags=["auth", "security"])
t2 = Task(id=1, title="Implement OAuth login", priority="High", tags=["auth", "security"])

# 2. Auto-generated __repr__ (beautiful readable output)
print("t1:", t1)

# 3. Auto-generated value equality (__eq__)
print("Are t1 and t2 equal by value?", t1 == t2)  # True!

# 4. Direct property access and mutation
t1.completed = True
print("After completion:", t1.completed)

## 4. Modern File I/O with `pathlib`
Python's `pathlib.Path` replaces legacy string-based path manipulations (`os.path.join`) with an object-oriented API using the slash operator `/`.

### Always Anchor to `Path(__file__).parent` (or `Path.cwd()` in notebooks)
Relative paths like `open("data.json")` depend on where your terminal was opened from. Anchoring paths ensures consistency across all environments.

In [ ]:
# Working with pathlib
base_dir = Path.cwd()
sandbox_dir = base_dir / "scratch_demo"
sample_file = sandbox_dir / "demo.txt"

# Create directory safely (mkdir -p equivalent)
sandbox_dir.mkdir(parents=True, exist_ok=True)

# Quick text write & read
sample_file.write_text("Hello from modern Python pathlib!", encoding="utf-8")

print("File exists:", sample_file.exists())
print("File name:", sample_file.name)
print("File extension:", sample_file.suffix)
print("Read content:", sample_file.read_text(encoding="utf-8"))

## 5. Context Managers & The `with` Statement
The **`with`** statement guarantees that resources (like file handles or database connections) are closed immediately and deterministically when execution leaves the block—even if an error is thrown.

In [ ]:
log_file = sandbox_dir / "app.log"

# Writing with automatic file closing
with open(log_file, mode="w", encoding="utf-8") as f:
    f.write("[INFO] Application starting up\n")
    f.write("[INFO] Database connection established\n")
# At this line, the file handle is 100% closed!

# Reading line by line
with open(log_file, mode="r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        print(f"Line {line_num}: {line.strip()}")

## 6. JSON Serialization & Persistence
Python's `json` module provides 4 functions:
- `json.dump(obj, file)`: Serializes object directly to a file stream.
- `json.load(file)`: Deserializes directly from a file stream.
- `json.dumps(obj)`: Serializes object to an in-memory JSON **s**tring (`JSON.stringify`).
- `json.loads(str)`: Deserializes from a JSON **s**tring (`JSON.parse`).

In [ ]:
from dataclasses import asdict

tasks_to_save = [
    asdict(Task(id=1, title="Configure DNS", completed=True, priority="High")),
    asdict(Task(id=2, title="Build GraphQL schema", completed=False, priority="Medium", tags=["api"]))
]

json_path = sandbox_dir / "tasks.json"

# 1. Save to JSON file
with open(json_path, mode="w", encoding="utf-8") as f:
    json.dump(tasks_to_save, f, indent=2)

# 2. Read back from JSON file
with open(json_path, mode="r", encoding="utf-8") as f:
    loaded_raw = json.load(f)

# Reconstruct Task dataclasses
reconstructed_tasks = [Task(**item) for item in loaded_raw]
print("Reconstructed Tasks:")
for t in reconstructed_tasks:
    print(" -", t)

## 7. Exception Handling & Custom Domain Exceptions
Structured error handling in Python uses `try`, `except`, `else`, and `finally`.

### Creating Custom Exceptions:
Inherit from `Exception` to build clear application error boundaries.

In [ ]:
# Custom domain exceptions
class TaskAppError(Exception):
    """Base exception for all Task Application errors."""
    pass

class TaskNotFoundError(TaskAppError):
    def __init__(self, task_id: int):
        self.task_id = task_id
        super().__init__(f"Task with ID {task_id} does not exist in store.")

# Function raising custom exception
def find_task_by_id(task_id: int, tasks: list[Task]) -> Task:
    for t in tasks:
        if t.id == task_id:
            return t
    raise TaskNotFoundError(task_id)

# Handling specific exceptions
try:
    found = find_task_by_id(999, reconstructed_tasks)
except TaskNotFoundError as err:
    print(f"Expected Domain Error Handled: {err} (Target ID: {err.task_id})")

## 8. Variadic Arguments: `*args` and `**kwargs`
- `*args`: Collects extra positional arguments into a `tuple` (like JS rest parameter `...args`).
- `**kwargs`: Collects extra keyword arguments into a `dict`.
- `*` and `**` can also unpack iterables and dicts when calling functions.

In [ ]:
def dispatch_request(action: str, *args, **kwargs):
    print(f"Action: {action}")
    print(f"Positional args (tuple): {args}")
    print(f"Keyword args (dict): {kwargs}")

dispatch_request("SAVE_RECORD", "v1", "sync", user_id=42, ip="10.0.0.1")

# Unpacking at call sites
def configure_client(host: str, port: int, ssl: bool = True):
    print(f"Client connecting to {host}:{port} (SSL: {ssl})")

config_dict = {"host": "api.internal", "port": 8443, "ssl": True}
configure_client(**config_dict)  # Unpacks dictionary into keyword arguments

## 9. Introduction to Decorators
A decorator is simply a function that takes another function as an argument, adds behavior, and returns the wrapped function.

Writing `@my_decorator` above a function definition is syntax sugar for:  
`my_function = my_decorator(my_function)`

In [ ]:
import functools
import time

def execution_logger(func):
    @functools.wraps(func)  # Preserves func.__name__ and docstring
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        print(f"--> Starting execution of '{func.__name__}'")
        result = func(*args, **kwargs)
        duration = (time.perf_counter() - start_time) * 1000
        print(f"<-- Finished '{func.__name__}' in {duration:.2f}ms")
        return result
    return wrapper

@execution_logger
def compute_task_analytics(tasks: list[Task]) -> dict:
    """Calculates analytics summary over task records."""
    time.sleep(0.02)  # Simulate analytical processing
    total = len(tasks)
    completed = sum(1 for t in tasks if t.completed)
    return {"total": total, "completed": completed, "completion_rate": f"{(completed/total)*100:.1f}%" if total else "0%"}

stats = compute_task_analytics(reconstructed_tasks)
print("Analytics Result:", stats)

## 10. Key Gotchas: Mutable Class Attributes
Defining mutable default values directly in the class body shares that object across **all** instances of the class.

In [ ]:
# THE BUGGY PATTERN:
class BuggyStore:
    items = []  # Class attribute (shared by all instances!)

s1 = BuggyStore()
s2 = BuggyStore()
s1.items.append("Shared item")
print("BuggyStore s2.items:", s2.items)  # ['Shared item'] -> BUG!

# THE CORRECT PATTERN:
class SafeStore:
    def __init__(self):
        self.items = []  # Instance attribute (isolated)

s3 = SafeStore()
s4 = SafeStore()
s3.items.append("Isolated item")
print("SafeStore s4.items:", s4.items)  # [] -> Completely isolated!

---
## 11. Practice Challenge: Building a Structured Task Service
### Objective:
Create a modular Task Service that brings together dataclasses, file persistence, and custom exceptions:
1. Define a `Task` dataclass with fields: `id: int`, `title: str`, `completed: bool = False`, `priority: str = "Normal"`, and `tags: list[str]`.
2. Implement a `TaskManagerService` class initialized with a `storage_path: Path`.
3. Implement methods: `add_task(title, priority, tags)`, `get_task(id)`, `mark_completed(id)`, `save_to_file()`, and `load_from_file()`.
4. Raise a custom `TaskNotFoundError` if querying or modifying a non-existent task ID.

In [ ]:
# TODO: Write your implementation here...


---
## 12. Challenge Solution

In [ ]:
# Reference Solution
from dataclasses import dataclass, field, asdict
from pathlib import Path
import json

class TaskNotFoundError(Exception):
    pass

@dataclass
class TaskItem:
    id: int
    title: str
    completed: bool = False
    priority: str = "Normal"
    tags: list[str] = field(default_factory=list)

class TaskManagerService:
    def __init__(self, file_path: Path):
        self.file_path = file_path
        self.tasks: list[TaskItem] = []
        self._next_id = 1
        self.load_from_file()

    def add_task(self, title: str, priority: str = "Normal", tags: list[str] | None = None) -> TaskItem:
        task = TaskItem(
            id=self._next_id,
            title=title,
            priority=priority,
            tags=tags if tags is not None else []
        )
        self.tasks.append(task)
        self._next_id += 1
        self.save_to_file()
        return task

    def get_task(self, task_id: int) -> TaskItem:
        for t in self.tasks:
            if t.id == task_id:
                return t
        raise TaskNotFoundError(f"Task with ID {task_id} not found.")

    def mark_completed(self, task_id: int) -> TaskItem:
        task = self.get_task(task_id)
        task.completed = True
        self.save_to_file()
        return task

    def save_to_file(self) -> None:
        self.file_path.parent.mkdir(parents=True, exist_ok=True)
        with open(self.file_path, "w", encoding="utf-8") as f:
            raw_list = [asdict(t) for t in self.tasks]
            json.dump(raw_list, f, indent=2)

    def load_from_file(self) -> None:
        if not self.file_path.exists():
            self.tasks = []
            return
        with open(self.file_path, "r", encoding="utf-8") as f:
            raw_list = json.load(f)
            self.tasks = [TaskItem(**item) for item in raw_list]
            if self.tasks:
                self._next_id = max(t.id for t in self.tasks) + 1

# Test the service
storage = Path.cwd() / "scratch_demo" / "service_tasks.json"
service = TaskManagerService(storage)

t_new = service.add_task("Build FastAPI endpoints", priority="High", tags=["api", "fastapi"])
print("Added Task:", t_new)
service.mark_completed(t_new.id)
print("Updated Task:", service.get_task(t_new.id))
print("All Tasks in Service:", len(service.tasks))

---
## 13. Day 2 Recap & Looking Ahead to Day 3

### What We Mastered Today:
1. **Modules & Imports:** Clean separation of concerns with standard libraries and local packages.
2. **Dataclasses:** Declarative data models with `@dataclass` and auto-generated equality / string formatting.
3. **Safe File I/O & JSON:** Object-oriented pathing with `pathlib.Path`, `with` context managers, and `json.dump/load`.
4. **Exceptions & Decorators:** Hierarchical custom error boundaries, variadic args (`*args`, `**kwargs`), and higher-order decorators.

### Looking Ahead to Day 3: Web Integration & Async Python
Tomorrow we take our Python applications onto the web:
- Consuming REST APIs with **`HTTPX`**
- Modern **Asynchronous Python** (`async def`, `await`, `asyncio.gather`)
- Understanding how Python coroutines map to JavaScript Promises
- Streaming data with **Generators (`yield`)**
- Web scraping and HTML parsing with **BeautifulSoup4**